# 2 – RAG: Fragen an die Dokumente stellen (mit LangGraph)

Dieses Notebook baut eine **Retrieval-Augmented Generation**-Pipeline als **LangGraph**:

1. **Retrieve** – Relevante Chunks aus der Vektordatenbank holen
2. **Generate** – Ein LLM formuliert aus den Chunks eine Antwort

Embedding- und LLM-Backend werden **ausschließlich in der Konfigurationszelle**
umgeschaltet – lokal (LM Studio / HuggingFace) oder in der Cloud (DeepInfra / OpenRouter).

> **Voraussetzung:** Notebook 1 (Indexing) muss vorher ausgeführt worden sein.

## Benötigte Pakete

```bash
pip install -U langchain-core langchain-openai langchain-chroma langgraph gradio
# nur bei EMBEDDING_BACKEND = "huggingface":
pip install -U "langchain-huggingface[full]"
```

## Konfiguration

Wortgleich mit Notebook 1. Abschnitte, die dieses Notebook nicht braucht, stören hier nicht.

In [ ]:
# ============================================================
#  KONFIGURATION – in allen Notebooks der Reihe identisch
# ============================================================

# --- 1) Embedding-Backend -----------------------------------
#     Indexierung und Retrieval MÜSSEN dasselbe Backend nutzen.
EMBEDDING_BACKEND = "lmstudio"        # "lmstudio" | "huggingface"

# --- 2) LLM-Backend (ab Notebook 2) --------------------------
LLM_BACKEND = "lmstudio"              # "lmstudio" | "deepinfra" | "openrouter"

# --- 3) API-Keys der Cloud-Anbieter --------------------------
#     Nur ausfüllen, wenn das jeweilige Backend genutzt wird.
#     Bitte den Key für dich behalten: nicht weitergeben, nicht committen,
#     nicht in geteilten Notebooks stehen lassen.
DEEPINFRA_API_KEY  = ""
OPENROUTER_API_KEY = ""

# --- 4) Pfade ------------------------------------------------
DOC_SOURCE_DIR = "./documents"        # Quell-Dokumente (PDF + DOCX)
DB_DIR         = "./chroma_db"        # Vektordatenbank
BM25_DIR       = "./bm25_index"       # Lexikalischer Index
MODEL_PATH     = "./models"           # Modell-Cache (nur HuggingFace)
MANIFEST_PATH  = "./index_manifest.json"

COLLECTION_NAME = "langchain"         # muss in Notebook 1 und 2 gleich sein

# --- 5) Chunking (Notebook 1) --------------------------------
#     Ändert man das hier, muss der komplette Index neu gebaut werden.
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128                   # ~25 % Überlappung – gut für lange deutsche Sätze

# --- 6) Welche Indizes bauen? (Notebook 1) -------------------
BUILD_VECTOR_INDEX = True             # semantisch, braucht das Embedding-Modell
BUILD_BM25_INDEX   = True             # lexikalisch, braucht kein Modell

# --- 7) Retrieval (ab Notebook 2) ----------------------------
TOP_K = 10                            # Anzahl der Chunks pro Frage

# ============================================================
#  Backend-Details – normalerweise unverändert lassen
# ============================================================

LM_STUDIO_URL = "http://localhost:1234/v1"

EMBEDDING_MODELS = {
    "huggingface": "intfloat/multilingual-e5-large-instruct",
    "lmstudio":    "text-embedding-multilingual-e5-large-instruct",
}

LLM_CONFIG = {
    "lmstudio": {
        "base_url": LM_STUDIO_URL,
        "model":    "qwen/qwen3.5-9b",
        "api_key":  "lm-studio",                  # LM Studio prüft den Key nicht
    },
    "deepinfra": {
        "base_url": "https://api.deepinfra.com/v1/openai",
        "model":    "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "api_key":  DEEPINFRA_API_KEY,
    },
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "model":    "meta-llama/llama-3.3-70b-instruct",
        "api_key":  OPENROUTER_API_KEY,
    },
}

print(f"Embeddings: {EMBEDDING_BACKEND}  |  LLM: {LLM_BACKEND}")

## Imports

In [ ]:
import os
import json
import base64
from typing import List, TypedDict

import gradio as gr
from IPython import display

from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

## Womit wurde der Index gebaut?

Notebook 1 hinterlegt ein Manifest. Die folgende Zelle **gibt es nur aus** – sie prüft
nichts und bricht nichts ab.

> ⚠️ Vergleiche `embedding_backend` und `embedding_modell` mit deiner Konfiguration oben.
> Stimmen sie nicht überein, liegen Dokument- und Frage-Vektoren in verschiedenen Räumen
> und die Treffer sind Zufall.

In [ ]:
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, encoding="utf-8") as f:
        manifest = json.load(f)

    print(f"📝 Index-Manifest ({MANIFEST_PATH}):")
    for key, value in manifest.items():
        print(f"   {key:<18} {value}")
else:
    print(f"ℹ️  Kein Manifest unter '{MANIFEST_PATH}' gefunden.")
    print("   Der Index stammt vermutlich aus einem älteren Lauf von Notebook 1.")

## Embedding-Modell

Identisch zu Notebook 1: E5 erwartet die Präfixe `passage:` beim Indexieren und
`query:` beim Suchen, die der Wrapper automatisch setzt.

In [ ]:
class E5OpenAIEmbeddings(OpenAIEmbeddings):
    """E5-Präfixe für OpenAI-kompatible Endpunkte (LM Studio)."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return super().embed_documents(["passage: " + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query("query: " + text)


def build_embeddings():
    """Erzeugt das Embedding-Modell passend zu EMBEDDING_BACKEND."""
    if EMBEDDING_BACKEND == "lmstudio":
        return E5OpenAIEmbeddings(
            model=EMBEDDING_MODELS["lmstudio"],
            api_key="lm-studio",
            base_url=LM_STUDIO_URL,
            check_embedding_ctx_length=False,
        )

    if EMBEDDING_BACKEND == "huggingface":
        # Import erst hier, damit LM-Studio-Nutzer das Paket nicht brauchen
        from langchain_huggingface import HuggingFaceEmbeddings

        class E5HuggingFaceEmbeddings(HuggingFaceEmbeddings):
            """E5-Präfixe für lokal geladene sentence-transformers-Modelle."""

            def embed_documents(self, texts: list[str]) -> list[list[float]]:
                return super().embed_documents(["passage: " + t for t in texts])

            def embed_query(self, text: str) -> list[float]:
                return super().embed_query("query: " + text)

        return E5HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODELS["huggingface"],
            cache_folder=MODEL_PATH,
        )

    raise ValueError(f"Unbekanntes EMBEDDING_BACKEND: {EMBEDDING_BACKEND}")

embeddings = build_embeddings()
print(f"✅ Embedding-Modell bereit ({EMBEDDING_BACKEND}).")

## Vektordatenbank laden

In [ ]:
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=DB_DIR,
    embedding_function=embeddings,
)

print(f"✅ Vektordatenbank geladen – {vectorstore._collection.count()} Chunks verfügbar.")

## LLM konfigurieren

Das Sprachmodell ist frei wählbar: es formuliert nur die Antwort und berührt den Index nicht.

In [ ]:
if LLM_BACKEND not in LLM_CONFIG:
    raise ValueError(f"Unbekanntes LLM_BACKEND: {LLM_BACKEND}")

cfg = LLM_CONFIG[LLM_BACKEND]

if not cfg["api_key"]:
    raise ValueError(f"Kein API-Key für '{LLM_BACKEND}' – bitte in der Konfigurationszelle eintragen.")

llm = ChatOpenAI(
    model=cfg["model"],
    api_key=cfg["api_key"],
    base_url=cfg["base_url"],
    max_tokens=5000,
    temperature=0,          # Für RAG: keine Kreativität, sondern Fakten
)

print(f"✅ LLM bereit: {cfg['model']} ({LLM_BACKEND}).")

## LangGraph: State, Nodes & Graph definieren

Ein LangGraph besteht aus:
- **State** – die Datenstruktur, die durch den Graphen fließt
- **Nodes** – Funktionen, die den State verändern
- **Edges** – die Verbindungen zwischen den Nodes

In [ ]:
# --- STATE ---

class GraphState(TypedDict):
    question:    str
    context:     List[str]
    metadata:    List[dict]
    answer:      str
    token_usage: dict

In [ ]:
# --- NODE: RETRIEVE ---

def retrieve(state: GraphState) -> dict:
    """Sucht die relevantesten Chunks zur Frage aus der Vektordatenbank."""
    print("--- RETRIEVE ---")
    docs = vectorstore.similarity_search(state["question"], k=TOP_K)

    context  = []
    metadata = []

    for i, doc in enumerate(docs):
        context.append(doc.page_content)

        # Metadaten aufbereiten: Pfad kürzen, Seitenzahl korrigieren (PDF zählt ab 0)
        source_file = os.path.basename(doc.metadata.get("source", "Unbekannt"))
        page_num    = doc.metadata.get("page", 0) + 1

        metadata.append({"id": i + 1, "source": source_file, "page": page_num})

    return {"context": context, "metadata": metadata}

In [ ]:
# --- NODE: GENERATE ---

PROMPT_TEMPLATE = ChatPromptTemplate.from_template("""\
Du bist ein präziser Assistent. Beantworte die Frage NUR basierend auf dem KONTEXT.

REGELN:
1. Verweise im Text deiner Antwort auf die Abschnitte, z.B. [1] oder [Quelle: Datei.pdf, S. 5].
2. Wenn die Info nicht im Kontext ist, sag es offen.
3. Erfinde KEINE Fakten.

KONTEXT:
{context}

FRAGE: {question}
""")


def generate(state: GraphState) -> dict:
    """Erzeugt eine Antwort auf Basis der gefundenen Kontext-Chunks."""
    print("--- GENERATE ---")

    # Kontext mit Quellenangaben formatieren, damit das LLM zuordnen kann
    formatted_context = ""
    for i, text in enumerate(state["context"]):
        meta = state["metadata"][i]
        formatted_context += (
            f"\n--- ABSCHNITT {meta['id']} "
            f"(Quelle: {meta['source']}, Seite {meta['page']}) ---\n"
            f"{text}\n"
        )

    chain    = PROMPT_TEMPLATE | llm
    response = chain.invoke({"context": formatted_context, "question": state["question"]})
    usage    = response.response_metadata.get("token_usage", {})

    return {"answer": response.content, "token_usage": usage}

In [ ]:
# --- GRAPH ZUSAMMENBAUEN ---

workflow = StateGraph(GraphState)

# Knoten registrieren
workflow.add_node("retrieve_node", retrieve)
workflow.add_node("generate_node", generate)

# Kanten definieren
workflow.add_edge(START, "retrieve_node")
workflow.add_edge("retrieve_node", "generate_node")
workflow.add_edge("generate_node", END)

# Graph kompilieren
app = workflow.compile()

## Graph visualisieren

Hier wird der kompilierte Graph als **Mermaid-Diagramm** dargestellt.  
Schau dir das Diagramm genau an – was fällt auf?

In [ ]:
def display_graph(graph_app):
    """Zeigt den LangGraph als Mermaid-Diagramm an und speichert die Syntax."""
    mermaid_code = graph_app.get_graph().draw_mermaid()

    # Diagramm im Notebook rendern (via mermaid.ink)
    encoded = base64.b64encode(mermaid_code.encode()).decode()
    display.display(display.Image(url=f"https://mermaid.ink/img/{encoded}"))

    # Mermaid-Syntax als Datei speichern (optional)
    with open("rag_graph.mmd", "w", encoding="utf-8") as f:
        f.write(mermaid_code)


display_graph(app)

## Gradio-Interface starten

In [ ]:
def chat_interface(question: str) -> str:
    """Verarbeitet eine Frage über den RAG-Graphen und formatiert die Ausgabe."""
    result = app.invoke({"question": question})

    # Antwort
    answer = result["answer"]

    # Token-Statistik
    usage = result.get("token_usage", {})
    token_info = (
        f"\n\n---\n📊 **Token-Statistik:**\n"
        f"- Input (Prompt): {usage.get('prompt_tokens', 'N/A')}\n"
        f"- Output (Antwort): {usage.get('completion_tokens', 'N/A')}\n"
        f"- Gesamt: {usage.get('total_tokens', 'N/A')}"
    )

    # Quellenverzeichnis
    sources = "\n\n**Verwendete Quellen:**"
    for meta in result["metadata"]:
        sources += f"\n- {meta['source']} (Seite {meta['page']})"

    return answer + token_info + sources


demo = gr.Interface(
    fn=chat_interface,
    inputs="text",
    outputs="text",
    title="Didaktik-RAG mit LangGraph",
    description="Stelle Fragen an deine indexierten PDF-Dokumente.",
    flagging_mode="never",
)

demo.launch()